<a href="https://colab.research.google.com/github/2anoFIAP/treinoCp1-DATASCIENSE/blob/main/cp1_NEMEC_Teste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('dataset_delivery_fraude_avaliacoes.csv', encoding='latin-1', sep=',')
df.columns = [c.replace('ï»¿', '') for c in df.columns]
df.head()

#Fase 1

media = df['preco_pedido'].mean()
mediana = df['preco_pedido'].median()
media_aparada = df['preco_pedido'].dropna()
resultado = stats.trim_mean(media_aparada, 0.1)

print(media)
print(mediana)
print(resultado)

#Respostas
#sim, os resultados se diferem de forma acentuada, devido provavelmente a existencia de outliers
#Sim, os outliers devem ser valores muito altos q fazem a media subir muito, por nao ser resitente a eles
#Mesmo q a media_aparada seja melhor q a media, a mediana ainda e a melhor por ser algo mais simles de ser mudavel e mais resistente a outliers

#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

#Fase 2
media_ponderada = (df['nota_cliente'] * df['preco_pedido']).sum() / df['preco_pedido'].sum()
#media simples para comparar
media_simples = df['nota_cliente'].mean()
print(media_simples)
print(media_ponderada)

#Respostas
#A media ponderada recebeu uma nota ligeiramente menor q a media normal, cerca de 0.01 oq nos mostra q independente do valor, as coisas estao normais, nao mudam a exp do cliente
#Um vies o preco nao tem nada haver com a nota, pois a correlacao entre eles e quase zero

#------------------------------------------------------------------------------------------------------------------------------------------------


#Fase 3
q1 = df['tempo_entrega_min'].quantile(0.25)
q3 = df['tempo_entrega_min'].quantile(0.75)
print(f"q1 {q1}")
print(f"q3 {q3}")

iqr = df['tempo_entrega_min'].quantile(0.75) - df['tempo_entrega_min'].quantile(0.25)
print(f"iqr {iqr}")

q1 = df["tempo_entrega_min"].quantile(0.25)
q3 = df["tempo_entrega_min"].quantile(0.75)
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
print(f"limite de baixo aqui: {limite_inferior}")
print(f"limite de cima: {limite_superior}")

outliers = df["tempo_entrega_min"][(df["tempo_entrega_min"] < limite_inferior) | (df["tempo_entrega_min"] > limite_superior)].tolist()
print(f"os outliers aqui: {outliers}")

#respostas
#existem 14 outliers
#SIm eles puxam a media pra cima devido a enorme quantidade de outliers e seus valores bem altos
#Na questao logistica eu removeria, para ver o tempo real q um entregador leva, mas na deteccao de fraudes, jamais, esses valores nos dao pistas muito boas para fraudes, como o tempo de entrega excessivo

#------------------------------------------------------------------------------------------------------------------------------------------------------------

#Fase 4
percentil_25 = df['preco_pedido'].quantile(0.25)
percentil_50 = df['preco_pedido'].quantile(0.50)
percentil_75 = df['preco_pedido'].quantile(0.75)

print(f"Percentil 25: {percentil_25}")
print(f"Percentil 50: {percentil_50}")
print(f"Percentil 75: {percentil_75}")

p90 = df['preco_pedido'].quantile(0.90)

print(f"O ponto de corte para os 10% mais caros é: R$ {p90:.2f}")

#respostas
#podemos definir um pedido caro de algumas formas, se estiver no corte dos 10% ele pode ser considarado caro, se tiver entre 75 e 90 pode ser considerado caro tbm e os acima de 90 muito caro

#--------------------------------------------------------------------------------------------------------------------------------------------

#FASE 5
#pega a lista real de cateogrias do dataset
categorias_reais = df['categoria'].unique()

for cat in categorias_reais:
    #aqui calcula todas as categorias
    df_temp = df[df['categoria'] == cat].copy()
    total_cat = len(df_temp)
    #se ticer zerada ele pula pra nao dar erro
    if total_cat == 0:
        continue

    # Calcula os quartis
    q1 = df_temp['taxa_entrega'].quantile(0.25)
    q3 = df_temp['taxa_entrega'].quantile(0.75)

    #saber as taxas
    def classificar(taxa):
        if taxa < q1: return 'Baixa'
        elif taxa > q3: return 'Alta'
        else: return 'Média'

    # Aplica a função e cria a coluna
    df_temp['classe_taxa'] = df_temp['taxa_entrega'].apply(classificar)

    # Calcula as probabilidades
    p_alta = len(df_temp[df_temp['classe_taxa'] == 'Alta']) / total_cat
    p_baixa = len(df_temp[df_temp['classe_taxa'] == 'Baixa']) / total_cat
    p_media = len(df_temp[df_temp['classe_taxa'] == 'Média']) / total_cat

    # Calcula os retornos
    media_taxa = df_temp['taxa_entrega'].mean()
    r_alta = 1.5 * media_taxa
    r_media = 1.0 * media_taxa
    r_baixa = 0.5 * media_taxa

    # Calcula o EV final
    ev = (p_alta * r_alta) + (p_media * r_media) + (p_baixa * r_baixa)

    print(f"Categoria: {cat} | EV: R$ {ev:.2f}")

#descobrir todas as categorias
#categorias_existentes = df['categoria'].unique()
#print(categorias_existentes)

#Respostas
#Cafe tem o maior EV
#provavelmente sim, pois nao faz o minimo de sentido o cafe ter o maior retorno esperado, sendo q tem categorias bem mais caras como comida JAPONESA q custa um rim

#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


#Fase 6
correlacao_pearsons = df['tempo_entrega_min'].corr(df['nota_cliente'])
print(f"Correlacao tempo x nota: {correlacao_pearsons}")

correlacao_pearsons2 = df['preco_pedido'].corr(df['nota_cliente'])
print(f"Correlacao preco x nota: {correlacao_pearsons2}")

correlacao_pearsons3 = df['distancia_km'].corr(df['tempo_entrega_min'])
print(f"Correlacao distancia x tempo: {correlacao_pearsons3}")
print('=======================================================================')

#Respostas
#Nao, mesmo q tenha um correlacao que mostra q atrasos acompanham notas baixas, demora nao e a unica causa, a matematica so da a pista da correlacao, mas precisamos analisar o contexto para afirmar a causa (falei bonito pqp)


#----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

#Fase 7
rest = df.groupby(['restaurante_id', 'nome_estabelecimento']).agg(
    n=('pedido_id', 'count'),
    nota_media=('nota_cliente', 'mean'),
    nota_std=('nota_cliente', 'std'),
    preco_medio=('preco_pedido', 'mean'),
    dist_media=('distancia_km', 'mean'),
    taxa_media=('taxa_entrega', 'mean'),
).reset_index()

suspeitos = rest.sort_values(['nota_media', 'nota_std'], ascending=[False, True])
print(suspeitos.head(5))

r18 = df[df['restaurante_id'] == 'R018']
print(r18[['item_principal','preco_pedido','distancia_km','taxa_entrega','nota_cliente']].to_string())
print(r18['item_principal'].value_counts())

#Resposta
#Restaurante: Império da Coxinha Nebulosa (R018)
#Evidências: nota média 4.69 com desvio padrão de apenas 0.25 em 26 pedidos (padrão artificial), todos os pedidos com itens baratos e repetidos (salgadinhos entre R$6-10), distância média de 0.89 km e taxa de entrega de R$2.53 (as menores do dataset). A combinação de notas quase perfeitas com variabilidade quase zero e pedidos homogêneos de baixo valor é incompatível com comportamento orgânico de clientes reais — fortíssima evidência de avaliações manipuladas.
#Nota mais alta ✅
#Desvio mais baixo ✅
#Preço mais barato ✅
#Distância mais curta ✅
#Taxa mais baixa ✅
#Mais pedidos que qualquer outro suspeito (26 vs 7 do segundo) ✅#

72.31084
54.11
62.33219999999999
3.6966
3.689308850512593
q1 30.0
q3 53.0
iqr 23.0
limite de baixo aqui: -4.5
limite de cima: 87.5
os outliers aqui: [164, 217, 189, 119, 133, 144, 147, 193, 172, 114, 91, 176, 118, 142]
Percentil 25: 28.575000000000003
Percentil 50: 54.11
Percentil 75: 95.8125
O ponto de corte para os 10% mais caros é: R$ 160.76
Categoria: Doces | EV: R$ 9.83
Categoria: Pizza | EV: R$ 8.61
Categoria: AÃ§aÃ­ | EV: R$ 8.42
Categoria: CafÃ© | EV: R$ 10.50
Categoria: Marmita | EV: R$ 8.80
Categoria: HambÃºrguer | EV: R$ 7.67
Categoria: Salgados | EV: R$ 7.32
Categoria: Japonesa | EV: R$ 9.05
Correlacao tempo x nota: -0.3894065343807447
Correlacao preco x nota: -0.015544670481985528
Correlacao distancia x tempo: 0.6242238455794581
   restaurante_id           nome_estabelecimento   n  nota_media  nota_std  \
17           R018   ImpÃ©rio da Coxinha Nebulosa  26    4.692308  0.249677   
48           R049      Lab do Gato Preto Supremo   7    4.500000  0.258199   
89           R